# Experiment: SpectralBio Block 1 - Baseline Alpha Regime Audit (H100)

Objective:
- Execute the real Block 1 notebook from the Strong Accept plan on Lightning AI Studio.
- Rebuild the stronger baseline surface for TP53 plus 1-2 key panel genes.
- Test whether `alpha=0.55` sits on a broad plateau rather than a fragile peak.
- Replace opaque regime naming with reviewer-facing labels and fixed selection rules.


## Block 1 Deliverables

- Stronger-baseline replay on `TP53`, `BRCA2`, and `MSH2` using the 5-model `ESM-1v` ensemble.
- Fixed-`0.55` comparison against full-surface alpha sweep, rank-based Borda fusion, and held-out logistic/isotonic fusion.
- Permutation audit to test whether covariance is only a rescaling artifact.
- Reviewer-facing artifacts for naming cleanup and threshold transparency.

## Runtime contract

- Target environment: **Lightning AI Studio**
- Target hardware: **H100**
- This notebook does **not** use Colab download helpers.
- All outputs are written under `New Notebooks/results/01_block1_baseline_alpha_regime_audit_h100/`.


In [ ]:
# Setup: imports, environment, and stable notebook identifiers
from __future__ import annotations

import json
import math
import os
import platform
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import MinMaxScaler

NOTEBOOK_SLUG = '01_block1_baseline_alpha_regime_audit_h100'
ACCOUNT_LABEL = os.environ.get('SPECTRALBIO_ACCOUNT_LABEL', 'SET_ACCOUNT_LABEL_HERE')
RUN_AT = datetime.now(timezone.utc).isoformat()
PLATEAU_TOLERANCE_AUC = float(os.environ.get('SPECTRALBIO_PLATEAU_TOLERANCE_AUC', '0.01'))
RUN_GENES = tuple(os.environ.get('SPECTRALBIO_BLOCK1_GENES', 'TP53,BRCA2,MSH2').split(','))
RUN_GENES = tuple(gene.strip().upper() for gene in RUN_GENES if gene.strip())
PRIMARY_GENES = tuple(gene for gene in RUN_GENES if gene in {'TP53', 'BRCA2'})
PERMUTATION_REPLICATES = int(os.environ.get('SPECTRALBIO_PERMUTATION_REPLICATES', '1000'))
RUN_3B_SHADOW = os.environ.get('SPECTRALBIO_RUN_3B_SHADOW', '').strip().lower() in {'1', 'true', 'yes'}
OVERWRITE = os.environ.get('SPECTRALBIO_OVERWRITE', '').strip().lower() in {'1', 'true', 'yes'}
SEED = int(os.environ.get('SPECTRALBIO_RANDOM_SEED', '42'))

def done(message: str) -> None:
    print(f'TERMINEI PODE SEGUIR - {message}')

print({
    'notebook_slug': NOTEBOOK_SLUG,
    'account_label': ACCOUNT_LABEL,
    'run_genes': RUN_GENES,
    'primary_genes': PRIMARY_GENES,
    'permutation_replicates': PERMUTATION_REPLICATES,
    'run_3b_shadow': RUN_3B_SHADOW,
    'overwrite': OVERWRITE,
    'seed': SEED,
    'python': sys.version.split()[0],
    'platform': platform.platform(),
})
done('Configuração inicial carregada.')


In [ ]:
# Helpers: subprocess, repo discovery, tables, and metrics
def run(command: list[str], cwd: Path | None = None) -> str:
    completed = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        check=True,
        text=True,
        encoding='utf-8',
        errors='replace',
        capture_output=True,
    )
    stdout = completed.stdout or ''
    stderr = completed.stderr or ''
    if stdout.strip():
        print(stdout.strip())
    if stderr.strip():
        print(stderr.strip())
    return stdout

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def looks_like_repo(path: Path) -> bool:
    return (path / 'src' / 'spectralbio').exists() and (path / 'notebooks').exists()

def find_repo_root() -> Path:
    env_repo = os.environ.get('SPECTRALBIO_REPO_ROOT', '').strip()
    candidates = []
    if env_repo:
        candidates.append(Path(env_repo))
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        Path('/teamspace/studios/this_studio/Stanford-Claw4s'),
        Path('/teamspace/studios/this_studio/SpectralBio'),
        Path('/content/Stanford-Claw4s'),
    ])
    for candidate in candidates:
        if looks_like_repo(candidate):
            return candidate.resolve()
    raise FileNotFoundError('Could not locate the SpectralBio repository root. Clone the repo and run the notebook from inside it, or set SPECTRALBIO_REPO_ROOT.')

def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))

def auc_safe(y_true, y_score) -> float:
    return float(roc_auc_score(np.asarray(y_true), np.asarray(y_score)))

def rank_borda_score(left: pd.Series, right: pd.Series) -> np.ndarray:
    left_rank = left.rank(method='average', ascending=True, pct=True)
    right_rank = right.rank(method='average', ascending=True, pct=True)
    return ((left_rank + right_rank) / 2.0).to_numpy(dtype=float)

def cv_combo_scores(frame: pd.DataFrame, feature_cols: list[str], label_col: str, seed: int = 42) -> dict:
    X = frame[feature_cols].to_numpy(dtype=float)
    y = frame[label_col].to_numpy(dtype=int)
    splitter = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=seed)
    logistic_scores = np.zeros(len(frame), dtype=float)
    isotonic_scores = np.zeros(len(frame), dtype=float)
    for train_idx, test_idx in splitter.split(X, y):
        scaler = MinMaxScaler()
        X_train = scaler.fit_transform(X[train_idx])
        X_test = scaler.transform(X[test_idx])
        model = LogisticRegression(max_iter=1000, random_state=seed)
        model.fit(X_train, y[train_idx])
        train_pred = model.predict_proba(X_train)[:, 1]
        test_pred = model.predict_proba(X_test)[:, 1]
        logistic_scores[test_idx] = test_pred
        iso = IsotonicRegression(out_of_bounds='clip')
        iso.fit(train_pred, y[train_idx])
        isotonic_scores[test_idx] = iso.predict(test_pred)
    return {
        'logistic_cv_score': logistic_scores,
        'isotonic_cv_score': isotonic_scores,
        'auc_logistic_cv': auc_safe(y, logistic_scores),
        'auc_isotonic_cv': auc_safe(y, isotonic_scores),
    }

def plateau_summary(alpha_rows: list[dict], fixed_alpha: float = 0.55, tolerance: float = 0.01) -> dict:
    alpha_df = pd.DataFrame(alpha_rows).sort_values('alpha').reset_index(drop=True)
    best_auc = float(alpha_df['auc'].max())
    best_alpha = float(alpha_df.loc[alpha_df['auc'].idxmax(), 'alpha'])
    plateau_df = alpha_df.loc[alpha_df['auc'] >= (best_auc - tolerance)].copy()
    fixed_row = alpha_df.loc[np.isclose(alpha_df['alpha'], fixed_alpha)]
    fixed_auc = float(fixed_row['auc'].iloc[0]) if not fixed_row.empty else float('nan')
    return {
        'best_alpha': best_alpha,
        'best_auc': best_auc,
        'fixed_alpha': fixed_alpha,
        'fixed_auc': fixed_auc,
        'delta_fixed_to_best_auc': float(best_auc - fixed_auc) if not math.isnan(fixed_auc) else float('nan'),
        'plateau_tolerance_auc': tolerance,
        'plateau_alpha_min': float(plateau_df['alpha'].min()),
        'plateau_alpha_max': float(plateau_df['alpha'].max()),
        'plateau_width': float(plateau_df['alpha'].max() - plateau_df['alpha'].min()),
        'fixed_alpha_in_plateau': bool(((plateau_df['alpha'] - fixed_alpha).abs() < 1e-9).any()),
    }

done('Helpers prontos.')


In [ ]:
# Resolve repository root and results directories for Lightning AI Studio
REPO_ROOT = find_repo_root()
RESULTS_ROOT = ensure_dir(REPO_ROOT / 'New Notebooks' / 'results' / NOTEBOOK_SLUG)
MANIFESTS_DIR = ensure_dir(RESULTS_ROOT / 'manifests')
TABLES_DIR = ensure_dir(RESULTS_ROOT / 'tables')
FIGURES_DIR = ensure_dir(RESULTS_ROOT / 'figures')
TEXT_DIR = ensure_dir(RESULTS_ROOT / 'text')
RUNTIME_DIR = ensure_dir(RESULTS_ROOT / 'runtime')
ZIP_PATH = REPO_ROOT / 'New Notebooks' / 'results' / f'{NOTEBOOK_SLUG}.zip'
repo_status = {
    'repo_root': str(REPO_ROOT),
    'head_commit': run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT).strip(),
    'head_subject': run(['git', 'log', '-1', '--pretty=%s'], cwd=REPO_ROOT).strip(),
    'branch': run(['git', 'branch', '--show-current'], cwd=REPO_ROOT).strip(),
}
display(pd.DataFrame([repo_status]))
done('Repositório e diretórios de saída confirmados.')


In [ ]:
# Ensure the local package imports cleanly inside Lightning AI Studio
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
try:
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        build_support_ranked_panel_manifest,
        create_accept_hardening_paths,
        run_esm1v_augmentation_suite,
        run_esm1v_permutation_audit,
        scan_clinvar_gene_support,
    )
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], cwd=str(REPO_ROOT))
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        build_support_ranked_panel_manifest,
        create_accept_hardening_paths,
        run_esm1v_augmentation_suite,
        run_esm1v_permutation_audit,
        scan_clinvar_gene_support,
    )
done('API do projeto importada.')


In [ ]:
# Configure the official Block 1 runtime
paths = create_accept_hardening_paths(repo_root=REPO_ROOT, output_root=RUNTIME_DIR)
config = AcceptHardeningConfig(
    random_seed=SEED,
    pair_alpha=0.55,
    alpha_step=0.05,
    checkpoint_every=10,
    bootstrap_replicates=1000,
    overwrite=OVERWRITE,
    render_figures=True,
)
ESM1V_MODEL_IDS = tuple(f'facebook/esm1v_t33_650M_UR90S_{index}' for index in range(1, 6))
runtime_manifest = {
    'account_label': ACCOUNT_LABEL,
    'run_at_utc': RUN_AT,
    'repo_status': repo_status,
    'run_genes': list(RUN_GENES),
    'primary_genes': list(PRIMARY_GENES),
    'permutation_replicates': PERMUTATION_REPLICATES,
    'run_3b_shadow': RUN_3B_SHADOW,
    'plateau_tolerance_auc': PLATEAU_TOLERANCE_AUC,
    'runtime_output_root': str(RUNTIME_DIR),
}
(MANIFESTS_DIR / 'runtime_manifest.json').write_text(json.dumps(runtime_manifest, indent=2) + '\n', encoding='utf-8')
display(pd.DataFrame([runtime_manifest]))
done('Runtime do Bloco 1 configurado.')


In [ ]:
# Build the fixed-threshold support scan and readable panel manifest
support_scan = scan_clinvar_gene_support(paths, config)
panel_manifest = build_support_ranked_panel_manifest(paths, config, support_scan)
selection_rule = panel_manifest.get('selection_rule', {})
selection_rule_rows = [
    {'field': 'support_scan_table', 'value': str(selection_rule.get('support_scan_table', ''))},
    {'field': 'min_total', 'value': int(selection_rule.get('min_total', 0))},
    {'field': 'min_per_class', 'value': int(selection_rule.get('min_per_class', 0))},
    {'field': 'max_additional_genes', 'value': int(selection_rule.get('max_additional_genes', 0))},
    {'field': 'ranking', 'value': ' | '.join(selection_rule.get('ranking', []))},
]
selection_rule_df = pd.DataFrame(selection_rule_rows)
selection_rule_df.to_csv(TABLES_DIR / 'selection_rule_table.csv', index=False)
display(selection_rule_df)
done('Support scan e regra de seleção congelados.')


In [ ]:
# Run the stronger-baseline augmentation suite for the Block 1 genes
augmentation_summary = run_esm1v_augmentation_suite(
    paths=paths,
    config=config,
    genes=RUN_GENES,
    esm1v_model_ids=ESM1V_MODEL_IDS,
    primary_genes=PRIMARY_GENES,
    run_3b_shadow=RUN_3B_SHADOW,
    panel_manifest=panel_manifest,
)
augmentation_long = pd.DataFrame(augmentation_summary.get('long_rows', []))
augmentation_long.to_csv(TABLES_DIR / 'augmentation_long_summary.csv', index=False)
display(augmentation_long)
done('Suíte ESM-1v + covariância concluída.')


In [ ]:
# Run the permutation audit to test whether the covariance term is only monotonic noise
permutation_summary = run_esm1v_permutation_audit(
    paths=paths,
    config=config,
    genes=RUN_GENES,
    permutation_replicates=PERMUTATION_REPLICATES,
)
permutation_df = pd.DataFrame(permutation_summary.get('rows', []))
permutation_df.to_csv(TABLES_DIR / 'permutation_audit_summary.csv', index=False)
display(permutation_df)
done('Permutation audit concluído.')


In [ ]:
# Build the Block 1 reviewer-facing comparison table from the generated augmentation tables
gene_frames = {}
for gene in RUN_GENES:
    table_path = RUNTIME_DIR / 'esm1v_augmentation' / gene.lower() / 'augmentation_table.csv'
    gene_frames[gene] = pd.read_csv(table_path)

audit_rows = []
alpha_rows = []
for gene, frame in gene_frames.items():
    frame = frame.copy()
    frame['borda_pair'] = rank_borda_score(frame['reference_frob_norm'], frame['esm1v_ll_mean_norm'])
    combo = cv_combo_scores(frame, ['reference_frob_norm', 'esm1v_ll_mean_norm'], 'label', seed=SEED)
    frame['logistic_cv_score'] = combo['logistic_cv_score']
    frame['isotonic_cv_score'] = combo['isotonic_cv_score']
    gene_frames[gene] = frame
    audit_rows.extend([
        {'gene': gene, 'score_family': 'esm1v_baseline', 'auc': auc_safe(frame['label'], frame['esm1v_ll_mean_norm'])},
        {'gene': gene, 'score_family': 'reference_pair_fixed_055', 'auc': auc_safe(frame['label'], frame['reference_pair_norm'])},
        {'gene': gene, 'score_family': 'augmented_pair_fixed_055', 'auc': auc_safe(frame['label'], frame['augmented_pair_fixed_055'])},
        {'gene': gene, 'score_family': 'borda_pair', 'auc': auc_safe(frame['label'], frame['borda_pair'])},
        {'gene': gene, 'score_family': 'logistic_cv_pair', 'auc': combo['auc_logistic_cv']},
        {'gene': gene, 'score_family': 'isotonic_cv_pair', 'auc': combo['auc_isotonic_cv']},
    ])
    alpha_rows_gene = augmentation_summary['genes'][gene]['alpha_sweep']
    alpha_info = plateau_summary(alpha_rows_gene, fixed_alpha=0.55, tolerance=PLATEAU_TOLERANCE_AUC)
    alpha_info['gene'] = gene
    alpha_rows.append(alpha_info)

audit_df = pd.DataFrame(audit_rows).sort_values(['gene', 'auc'], ascending=[True, False]).reset_index(drop=True)
alpha_df = pd.DataFrame(alpha_rows).sort_values('gene').reset_index(drop=True)
audit_df.to_csv(TABLES_DIR / 'baseline_alpha_audit_auc_table.csv', index=False)
alpha_df.to_csv(TABLES_DIR / 'alpha_plateau_summary.csv', index=False)
display(audit_df)
display(alpha_df)
done('Comparações de baseline e alpha consolidadas.')


In [ ]:
# Build the naming cleanup and reviewer-facing summary tables
glossary_df = pd.DataFrame([
    {
        'legacy_name': 'to_basic__AND__high_disagreement_q75',
        'proposed_name': 'high-covariance high-disagreement basic-substitution regime',
        'selection_explanation': 'Variants selected by fixed rescue, harm, and disagreement thresholds before any narrative inspection.',
    },
    {
        'legacy_name': 'support-ranked feasible panel',
        'proposed_name': 'performance-blind support-ranked feasible panel',
        'selection_explanation': 'Genes selected by predeclared support thresholds and feasibility filters, not by observed gain.',
    },
])
glossary_df.to_csv(TABLES_DIR / 'regime_glossary.csv', index=False)

reviewer_rows = []
for gene in RUN_GENES:
    gene_payload = augmentation_summary['genes'][gene]
    plateau_row = alpha_df.loc[alpha_df['gene'] == gene].iloc[0].to_dict()
    reviewer_rows.append({
        'gene': gene,
        'auc_esm1v_mean': float(gene_payload['auc_esm1v_mean']),
        'auc_reference_pair_fixed_055': float(gene_payload['auc_reference_pair_fixed_055']),
        'auc_augmented_pair_fixed_055': float(gene_payload['auc_augmented_pair_fixed_055']),
        'delta_augmented_vs_esm1v': float(gene_payload['delta_augmented_vs_esm1v']),
        'best_alpha_on_full_surface': float(gene_payload['best_alpha_on_full_surface']['alpha']),
        'best_alpha_auc': float(gene_payload['best_alpha_on_full_surface']['auc']),
        'fixed_alpha_in_plateau': bool(plateau_row['fixed_alpha_in_plateau']),
        'plateau_alpha_min': float(plateau_row['plateau_alpha_min']),
        'plateau_alpha_max': float(plateau_row['plateau_alpha_max']),
        'paired_delta_ci_2p5': float(gene_payload['pair_vs_esm1v_bootstrap_delta']['ci_2p5']),
        'paired_delta_ci_97p5': float(gene_payload['pair_vs_esm1v_bootstrap_delta']['ci_97p5']),
    })
reviewer_df = pd.DataFrame(reviewer_rows).sort_values('gene').reset_index(drop=True)
reviewer_df.to_csv(TABLES_DIR / 'reviewer_facing_summary_table.csv', index=False)
display(glossary_df)
display(reviewer_df)
done('Resumo reviewer-facing e glossário gerados.')


In [ ]:
# Render compact figures that can go straight into the paper draft or response letter
import matplotlib.pyplot as plt

for gene in RUN_GENES:
    alpha_rows_gene = pd.DataFrame(augmentation_summary['genes'][gene]['alpha_sweep']).sort_values('alpha')
    plt.figure(figsize=(7.2, 4.2))
    plt.plot(alpha_rows_gene['alpha'], alpha_rows_gene['auc'], marker='o', linewidth=2)
    plt.axvline(0.55, color='#b22222', linestyle='--', linewidth=1.5, label='released alpha=0.55')
    plt.title(f'{gene} full-surface alpha sweep')
    plt.xlabel('alpha on covariance term')
    plt.ylabel('AUC')
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{gene.lower()}_alpha_sweep.png', dpi=180, bbox_inches='tight')
    plt.close()

for gene in RUN_GENES:
    gene_df = audit_df.loc[audit_df['gene'] == gene].copy()
    plt.figure(figsize=(8.4, 4.6))
    plt.bar(gene_df['score_family'], gene_df['auc'], color=['#4666aa', '#5b8bd9', '#2f6f91', '#6f9e4a', '#c27d38', '#8f4db5'])
    plt.title(f'{gene} score-family comparison')
    plt.ylabel('AUC')
    plt.xticks(rotation=25, ha='right')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{gene.lower()}_score_family_auc.png', dpi=180, bbox_inches='tight')
    plt.close()

done('Figuras do Bloco 1 renderizadas.')


In [ ]:
# Write the final Block 1 summary as both JSON and Markdown
summary_payload = {
    'benchmark': 'SpectralBio Block 1 baseline alpha regime audit',
    'account_label': ACCOUNT_LABEL,
    'repo_status': repo_status,
    'run_genes': list(RUN_GENES),
    'reviewer_summary_rows': reviewer_df.to_dict(orient='records'),
    'alpha_plateau_rows': alpha_df.to_dict(orient='records'),
    'permutation_rows': permutation_df.to_dict(orient='records'),
    'selection_rule_rows': selection_rule_df.to_dict(orient='records'),
}
(MANIFESTS_DIR / 'block1_summary.json').write_text(json.dumps(summary_payload, indent=2) + '\n', encoding='utf-8')

summary_lines = [
    '# SpectralBio Block 1 Summary',
    '',
    f'- Account label: `{ACCOUNT_LABEL}`',
    f"- Repository HEAD: `{repo_status['head_commit'][:12]}` - {repo_status['head_subject']}",
    f'- Genes audited: `{", ".join(RUN_GENES)}`',
    f'- 3B shadow enabled: `{RUN_3B_SHADOW}`',
    '',
    '## Main takeaways',
    '',
]
for _, row in reviewer_df.iterrows():
    summary_lines.append(
        f"- {row['gene']}: ESM-1v AUC `{row['auc_esm1v_mean']:.4f}`, augmented pair AUC `{row['auc_augmented_pair_fixed_055']:.4f}`, best alpha `{row['best_alpha_on_full_surface']:.2f}`, fixed 0.55 in plateau `{bool(row['fixed_alpha_in_plateau'])}`"
    )
summary_lines.extend([
    '',
    '## Naming upgrades',
    '',
    '- `to_basic__AND__high_disagreement_q75` -> `high-covariance high-disagreement basic-substitution regime`',
    '- `support-ranked feasible panel` -> `performance-blind support-ranked feasible panel`',
])
(TEXT_DIR / 'block1_summary.md').write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
print((TEXT_DIR / 'block1_summary.md').read_text(encoding='utf-8'))
done('Resumo final do Bloco 1 escrito.')


In [ ]:
# Package the full result directory as a zip artifact for manual download from Lightning
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted(RESULTS_ROOT.rglob('*')):
        if file_path.is_file():
            archive.write(file_path, file_path.relative_to(RESULTS_ROOT.parent))

artifact_summary = {
    'results_root': str(RESULTS_ROOT),
    'zip_path': str(ZIP_PATH),
    'zip_size_bytes': ZIP_PATH.stat().st_size,
}
(MANIFESTS_DIR / 'artifact_summary.json').write_text(json.dumps(artifact_summary, indent=2) + '\n', encoding='utf-8')
display(pd.DataFrame([artifact_summary]))
done('Zip do Bloco 1 gerado.')


In [ ]:
# Final operator checkpoint for Lightning AI Studio
print('Notebook concluído.')
print('Resultados:', RESULTS_ROOT)
print('Zip final :', ZIP_PATH)
done('Bloco 1 pronto para revisão manual e envio do zip.')
